# Day 3 - NumPy Deep Dive (Remaining Topics, End-to-End)

> Continuation of Day 3. Notebook 1 covered NumPy Introduction & Array Creation.
> This notebook covers the remaining 5 topics in one file:
1. Indexing & Slicing
2. Boolean Masking & Filtering
3. Broadcasting
4. Vectorization vs Loops
5. NumPy in EDA Workflows

We continue using the **same 10-customer FD dataset** (as separate 1D NumPy arrays per column), so everything stays grounded in one familiar example.


## 0. Setup - Rebuilding Our Dataset

Since this is a new notebook, we recreate the same FD dataset arrays from Notebook 1 first.


In [1]:
import numpy as np

customer_id = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
age              = np.array([25, 34, 45, 29, 52, 23, 41, 38, 60, 31])
annual_income    = np.array([45.0, 62.0, np.nan, 38.5, 95.0, 30.0, 71.0, 55.0, 950.0, np.nan])
account_balance  = np.array([12.5, 45.2, 78.9, 5.0, 120.0, -2.5, 60.0, 0.0, 200.0, 8.0])
tenure_years     = np.array([1, 3, 8, 1, 15, 0, 6, 4, 20, 2])
num_products     = np.array([1, 2, 3, 1, 4, 1, 2, 2, 5, 1])
is_active = np.array([1, 1, 1, 0, 1, 0, 1, 0, 1, 1])
FD        = np.array([0, 1, 1, 0, 1, 0, 1, 0, 1, 0])
gender = np.array(["M", "F", "M", "F", "M", "F", "M", "F", "M", "F"])
city   = np.array(["Pune", "Mumbai", "Delhi", "Pune", "Bangalore",
                    "Mumbai", "Delhi", "Bangalore", "Pune", "Delhi"])

# 2D feature matrix (numeric columns only) - rows = customers, columns = features
feature_matrix = np.array([age, annual_income, account_balance, tenure_years, num_products]).T
print("feature_matrix shape:", feature_matrix.shape)
print(feature_matrix)


feature_matrix shape: (10, 5)
[[ 25.   45.   12.5   1.    1. ]
 [ 34.   62.   45.2   3.    2. ]
 [ 45.    nan  78.9   8.    3. ]
 [ 29.   38.5   5.    1.    1. ]
 [ 52.   95.  120.   15.    4. ]
 [ 23.   30.   -2.5   0.    1. ]
 [ 41.   71.   60.    6.    2. ]
 [ 38.   55.    0.    4.    2. ]
 [ 60.  950.  200.   20.    5. ]
 [ 31.    nan   8.    2.    1. ]]


## 1. Indexing & Slicing

### Introduction
- **Indexing** means accessing a single element using its position
- **Slicing** means accessing a range/subset of elements using `start:stop:step`
- Why do we need it?
  - Real datasets require selecting specific customers, specific features, or specific sub-ranges of data constantly during EDA

### Real-Life Analogy
- Think of our array like a **row of numbered lockers in a hallway**
- Indexing = opening **one specific locker** (e.g., locker number 3)
- Slicing = opening **a continuous range of lockers** at once (e.g., lockers 3 to 7)
- For 2D arrays (like our feature matrix), think of a **grid of lockers** arranged in rows and columns - you can pick a specific row, column, or a rectangular block

### Explanation
- 1D indexing: `arr[i]` -> single element; negative indices count from the end (`arr[-1]` = last element)
- 1D slicing: `arr[start:stop:step]` -> `stop` is **exclusive**
- 2D indexing: `matrix[row, col]` -> comma-separated, unlike nested Python lists (`list[row][col]`)
- 2D slicing: `matrix[row_start:row_stop, col_start:col_stop]`
- **Fancy indexing**: passing a **list/array of indices** instead of a single index or slice, to select multiple specific (non-contiguous) elements at once

### Syntax
```python
arr[3]              # single element at index 3
arr[-1]              # last element
arr[2:5]             # elements at index 2, 3, 4 (5 excluded)
arr[:3]              # first 3 elements
arr[::2]             # every 2nd element

matrix[1, 2]          # row 1, column 2
matrix[0:2, :]        # first 2 rows, all columns
matrix[:, 1]          # all rows, column 1 only

arr[[0, 2, 4]]        # fancy indexing - elements at positions 0, 2, 4
```


In [2]:
# Basic Example - 1D indexing and slicing on age
print("First customer's age:", age[0])
print("Last customer's age:", age[-1])
print("Ages of customers 2 to 4 (index 1:4):", age[1:4])
print("Every other customer's age:", age[::2])


First customer's age: 25
Last customer's age: 31
Ages of customers 2 to 4 (index 1:4): [34 45 29]
Every other customer's age: [25 45 52 41 60]


**Line by line explanation:**
- `age[0]` -> first element (customer 1's age)
- `age[-1]` -> negative indexing counts from the end, so this is the last customer's age
- `age[1:4]` -> a slice from index 1 up to (not including) index 4, giving 3 elements
- `age[::2]` -> `start` and `stop` omitted means "whole array"; `step=2` means "every second element"


In [3]:
# Intermediate Example - 2D indexing and slicing on feature_matrix
# Recall columns are: [age, annual_income, account_balance, tenure_years, num_products]

print("Customer at row 2, feature 'account_balance' (column index 2):", feature_matrix[2, 2])
print("\nFirst 3 customers, all features:\n", feature_matrix[0:3, :])
print("\nAll customers, only 'annual_income' column (index 1):\n", feature_matrix[:, 1])


Customer at row 2, feature 'account_balance' (column index 2): 78.9

First 3 customers, all features:
 [[25.  45.  12.5  1.   1. ]
 [34.  62.  45.2  3.   2. ]
 [45.   nan 78.9  8.   3. ]]

All customers, only 'annual_income' column (index 1):
 [ 45.   62.    nan  38.5  95.   30.   71.   55.  950.    nan]


**Line by line explanation:**
- `feature_matrix[2, 2]` -> row index 2 (3rd customer), column index 2 (`account_balance`) - notice the comma-separated syntax, different from nested Python lists which would need `matrix[2][2]`
- `feature_matrix[0:3, :]` -> rows 0,1,2 (first 3 customers), `:` for columns means "all columns"
- `feature_matrix[:, 1]` -> `:` for rows means "all rows", column index 1 selects just `annual_income` for everyone


In [4]:
# Real-world Example - Fancy indexing to select specific customers by ID
selected_positions = [0, 3, 8]   # positions for customer_id 1, 4, 9

print("Selected customers' IDs:", customer_id[selected_positions])
print("Selected customers' ages:", age[selected_positions])
print("Selected customers' account balances:", account_balance[selected_positions])


Selected customers' IDs: [1 4 9]
Selected customers' ages: [25 29 60]
Selected customers' account balances: [ 12.5   5.  200. ]


**Line by line explanation:**
- `selected_positions = [0, 3, 8]` -> a list of specific, non-contiguous positions we want (not a continuous range, so slicing alone can't do this)
- `customer_id[selected_positions]` -> **fancy indexing**: passing a list/array of positions returns an array with just those elements, in that order
- This is extremely useful in EDA for pulling out specific rows of interest (e.g., flagged customers, sampled customers)


### Internal Working
- **Basic slicing** (`arr[2:5]`) returns a **view**, not a copy - it points to the **same underlying memory** as the original array. Modifying a slice modifies the original array too
- **Fancy indexing** (`arr[[0,2,4]]`) always returns a **copy** - a completely new array with its own memory
- This distinction is a very common source of subtle bugs and a favorite interview question

### Time & Space Complexity
- Single element indexing (`arr[i]`) -> Time: O(1), Space: O(1) - direct memory address calculation
- Basic slicing (`arr[a:b]`) -> Time: O(1) to create the view, but O(k) if the resulting slice of size k is later read/copied; Space: O(1) extra (since it's a view, not a copy)
- Fancy indexing (`arr[[i,j,k]]`) -> Time: O(k), Space: O(k), where k is the number of indices requested (since a new array is created)


In [5]:
# Demonstrating view vs copy behavior
age_slice = age[0:3]      # basic slicing -> VIEW
age_fancy = age[[0, 1, 2]]  # fancy indexing -> COPY

age_slice[0] = 999   # modifying the slice/view
print("Original age array after modifying the SLICE:", age)

# reset for the next demonstration
age[0] = 25

age_fancy[0] = 888   # modifying the fancy-indexed copy
print("Original age array after modifying the FANCY-INDEXED copy:", age)


Original age array after modifying the SLICE: [999  34  45  29  52  23  41  38  60  31]
Original age array after modifying the FANCY-INDEXED copy: [25 34 45 29 52 23 41 38 60 31]


**Line by line explanation:**
- Modifying `age_slice[0]` (from basic slicing) **also changes** the original `age` array - proving slicing returns a view sharing memory
- Modifying `age_fancy[0]` (from fancy indexing) does **NOT** affect the original `age` array - proving fancy indexing returns an independent copy

> Common mistake alert: Assuming a slice is a safe, independent copy - if you need an actual copy, use `.copy()` explicitly: `age_slice = age[0:3].copy()`.

### Common Mistakes
- Forgetting that basic slices are views, causing accidental modification of original data
- Confusing 2D indexing syntax `matrix[row, col]` with nested list syntax `matrix[row][col]` (both technically work on NumPy arrays, but `[row, col]` is the idiomatic, faster way)
- Off-by-one errors, forgetting that `stop` is exclusive in slices

### Best Practices
- Use `.copy()` explicitly whenever you need an independent copy of a slice
- Prefer `matrix[row, col]` syntax over `matrix[row][col]` for 2D NumPy arrays - more efficient (avoids creating an intermediate array)
- Use fancy indexing when you need specific, non-contiguous elements; use slicing for contiguous ranges


## 2. Boolean Masking & Filtering

### Introduction
- **Boolean masking** means creating an array of `True`/`False` values based on a condition, then using that mask to filter data
- Why do we need it?
  - This is the primary way to filter rows/values based on conditions in NumPy - much faster than writing manual loops with `if` statements

### Real-Life Analogy
- Imagine a teacher checking a class of 10 students and putting a **sticky note (True/False)** on each student's desk based on whether they passed an exam
- Then, the teacher can quickly "collect only the True desks" to get the list of students who passed - that's exactly what boolean masking does

### Explanation
- Any condition applied to a NumPy array (e.g., `age > 40`) returns a **boolean array** of the same shape, called a **mask**
- Using this mask inside `array[mask]` returns only the elements where the mask is `True`
- Multiple conditions can be combined using `&` (and), `|` (or), `~` (not) - each condition must be wrapped in parentheses
- `np.where(condition, value_if_true, value_if_false)` -> a vectorized if-else, returns a new array
- `np.select(conditions_list, choices_list)` -> vectorized multi-condition if-elif-else

### Syntax
```python
mask = age > 40                 # boolean array
filtered = age[mask]            # elements where mask is True

# Combined conditions
filtered2 = age[(age > 30) & (FD == 1)]   # AND
filtered3 = age[(city == "Pune") | (city == "Delhi")]   # OR

result = np.where(condition, value_if_true, value_if_false)
result2 = np.select([cond1, cond2], [choice1, choice2], default=fallback)
```


In [6]:
# Basic Example - simple boolean mask
mask = age > 40
print("Boolean mask (age > 40):", mask)
print("Filtered ages (only > 40):", age[mask])
print("Corresponding customer IDs:", customer_id[mask])


Boolean mask (age > 40): [False False  True False  True False  True False  True False]
Filtered ages (only > 40): [45 52 41 60]
Corresponding customer IDs: [3 5 7 9]


**Line by line explanation:**
- `age > 40` -> compares every element of `age` to 40, returns a same-shaped boolean array
- `age[mask]` -> filters `age`, keeping only positions where `mask` is `True`
- We also apply the **same mask** to `customer_id` - a key technique: masks generated from one column can filter any other column, since they share the same row positions


In [7]:
# Intermediate Example - combined conditions
# Customers who are both older than 30 AND have an FD
combined_mask = (age > 30) & (FD == 1)
print("Customers older than 30 with FD:")
print("IDs:", customer_id[combined_mask])
print("Ages:", age[combined_mask])

# Customers from Pune OR Delhi
location_mask = (city == "Pune") | (city == "Delhi")
print("\nCustomers from Pune or Delhi:", customer_id[location_mask])


Customers older than 30 with FD:
IDs: [2 3 5 7 9]
Ages: [34 45 52 41 60]

Customers from Pune or Delhi: [ 1  3  4  7  9 10]


**Line by line explanation:**
- `(age > 30) & (FD == 1)` -> each condition is evaluated separately (producing 2 boolean arrays), then combined element-wise with `&` (logical AND)
- Parentheses around each condition are **required** in NumPy - without them, Python's operator precedence causes errors
- `|` works the same way for OR conditions


In [8]:
# Real-world Example - handling our known NaN and outlier using masks

# Detect missing income values
nan_mask = np.isnan(annual_income)
print("Customers with missing income:", customer_id[nan_mask])

# Detect the income outlier using a threshold
outlier_mask = annual_income > 500
print("Customers with unusually high income:", customer_id[outlier_mask])

# Using np.where to create a new balance risk flag
balance_risk = np.where(account_balance < 0, "overdraft",
                 np.where(account_balance == 0, "empty", "healthy"))
print("\nBalance risk flags:", balance_risk)


Customers with missing income: [ 3 10]
Customers with unusually high income: [9]

Balance risk flags: ['healthy' 'healthy' 'healthy' 'healthy' 'healthy' 'overdraft' 'healthy'
 'empty' 'healthy' 'healthy']


**Line by line explanation:**
- `np.isnan(annual_income)` -> a special function that correctly detects `NaN` values (a plain `annual_income == np.nan` would NOT work, since `NaN != NaN` by definition - a classic NumPy gotcha)
- `annual_income > 500` -> a simple threshold-based mask that correctly isolates our planted outlier (customer 9)
- `np.where(...)` nested -> vectorized multi-branch logic, same pattern we used in the Pandas notebook, but here operating directly on NumPy arrays


### Internal Working
- `array[boolean_mask]` internally scans the mask and the array together, position by position, and copies only the elements where the mask is `True` into a **new array**
- Unlike basic slicing, **boolean masking always returns a copy**, not a view
- `np.isnan()` works by checking the special IEEE 754 floating point bit pattern reserved for NaN, rather than a normal value comparison - this is why `== np.nan` fails but `np.isnan()` works correctly

### Time & Space Complexity
- Creating a mask (`age > 40`) -> Time: O(n), Space: O(n) - one boolean value produced per element
- Applying a mask (`age[mask]`) -> Time: O(n) (must scan the whole array), Space: O(k) where k is the number of `True` values (since a new array is created)

### Common Mistakes
- Using `annual_income == np.nan` to detect missing values -> always returns `False`, even for actual NaN values; must use `np.isnan()`
- Forgetting parentheses around each condition when combining with `&`/`|`
- Using Python's `and`/`or` instead of `&`/`|` on NumPy arrays -> raises an error (Python's `and`/`or` only work on single boolean values, not arrays)

### Best Practices
- Always use `np.isnan()` (not `==`) to detect missing float values
- Always wrap each condition in parentheses when combining with `&`, `|`, `~`
- Use `np.where()` for simple 2-branch vectorized logic, and `np.select()` when you have 3+ conditions to keep code readable


## 3. Broadcasting

### Introduction
- **Broadcasting** is NumPy's way of performing operations between arrays of **different shapes**, by automatically "stretching" the smaller array to match the larger one, without actually copying data
- Why do we need it?
  - Lets us write simple code like `array + 5` instead of manually looping and adding 5 to every element

### Real-Life Analogy
- Imagine you want to give **every customer a flat bonus of ₹5** on their account balance
- Instead of walking up to each of the 10 customers individually and adding ₹5 one at a time, broadcasting lets you announce once: "everyone gets +5" - and it applies to all of them simultaneously

### Explanation
- When performing an operation between two arrays of different shapes, NumPy compares their shapes **from right to left**
- Two dimensions are "compatible" for broadcasting if:
  - They are equal, OR
  - One of them is 1 (it gets **stretched** to match the other)
- If shapes are incompatible after this comparison, NumPy raises an error

### Syntax
```python
arr + 5                     # scalar broadcast to every element
matrix + row_vector         # row_vector's shape (1, n) broadcasts across all rows
matrix + column_vector      # column_vector's shape (n, 1) broadcasts across all columns
```


In [9]:
# Basic Example - scalar broadcasting
bonus_balance = account_balance + 5
print("Original balances:   ", account_balance)
print("After +5 bonus:       ", bonus_balance)

# Scaling income by 10% (broadcasting with multiplication)
income_after_raise = annual_income * 1.10
print("\nIncome after 10% raise:", income_after_raise)


Original balances:    [ 12.5  45.2  78.9   5.  120.   -2.5  60.    0.  200.    8. ]
After +5 bonus:        [ 17.5  50.2  83.9  10.  125.    2.5  65.    5.  205.   13. ]

Income after 10% raise: [  49.5    68.2      nan   42.35  104.5    33.     78.1    60.5  1045.
     nan]


**Line by line explanation:**
- `account_balance + 5` -> the scalar `5` is conceptually "stretched" to match the shape of `account_balance` (10 elements), then added element-wise
- Under the hood, NumPy does NOT actually create 10 copies of `5` - it's smart about this (explained in Internal Working below)
- `annual_income * 1.10` -> same idea, multiplying every element by a scalar; notice the `NaN` values remain `NaN` (any arithmetic with NaN produces NaN, which is why we must handle NaN before analysis, not after)


In [10]:
# Intermediate Example - broadcasting a 1D array across a 2D matrix
# feature_matrix columns are: [age, annual_income, account_balance, tenure_years, num_products]

# Suppose we want to normalize each feature by subtracting per-feature minimums
feature_mins = np.array([20, 0, -5, 0, 0])   # one minimum per column (shape: (5,))

print("feature_matrix shape:", feature_matrix.shape)
print("feature_mins shape:  ", feature_mins.shape)

normalized = feature_matrix - feature_mins
print("\nFirst 3 rows after subtracting feature_mins:\n", normalized[0:3])


feature_matrix shape: (10, 5)
feature_mins shape:   (5,)

First 3 rows after subtracting feature_mins:
 [[ 5.  45.  17.5  1.   1. ]
 [14.  62.  50.2  3.   2. ]
 [25.   nan 83.9  8.   3. ]]


**Line by line explanation:**
- `feature_matrix` has shape `(10, 5)`, `feature_mins` has shape `(5,)`
- NumPy compares shapes right to left: last dimension `5` matches `5` -> compatible
- `feature_mins` (shape `(5,)`) is broadcast across all 10 rows, effectively as if it were shape `(10, 5)`, without actually duplicating memory 10 times
- Each column gets its own minimum subtracted - column 0 (age) has 20 subtracted from every row, column 1 (income) has 0 subtracted, etc.


In [11]:
# Real-world Example - normalizing income to a 0-1 scale using broadcasting (min-max scaling)
# Using nanmin/nanmax to safely ignore NaN values (covered fully in EDA section below)

income_min = np.nanmin(annual_income)
income_max = np.nanmax(annual_income)

income_normalized = (annual_income - income_min) / (income_max - income_min)
print("Min:", income_min, "| Max:", income_max)
print("\nNormalized income (0 to 1 scale):")
print(income_normalized)


Min: 30.0 | Max: 950.0

Normalized income (0 to 1 scale):
[0.01630435 0.03478261        nan 0.00923913 0.07065217 0.
 0.04456522 0.02717391 1.                nan]


**Line by line explanation:**
- `income_min` and `income_max` are **scalars** (single numbers)
- `(annual_income - income_min)` -> broadcasting: the scalar is subtracted from every element
- Dividing by `(income_max - income_min)` -> another scalar broadcast, scaling everything to a 0-1 range
- Because our outlier (950.0) defines `income_max`, notice how it compresses every other customer's normalized value very close to 0 - a great visual demonstration of **why outliers distort normalization**, and why they must be handled before scaling in real ML pipelines

### Internal Working
- Broadcasting does **not** physically copy the smaller array into a larger one in memory
- Instead, NumPy uses **stride tricks** - it reuses the same memory for the smaller array but tells the internal loop to "replay" it as many times as needed
- This makes broadcasting both fast (no extra memory for the stretched array) and memory efficient

### Time & Space Complexity
- Broadcasting itself adds no extra time/space overhead beyond the operation being performed
- The operation (e.g., `matrix - vector`) runs in O(n) or O(n*m) depending on shape, same as if shapes were already equal, but WITHOUT the extra O(n*m) memory that manually duplicating the smaller array would require

### Common Mistakes
- Assuming any two different shapes will broadcast - they only broadcast if dimensions are equal or one of them is 1; mismatched shapes raise a `ValueError`
- Forgetting NaN values propagate through broadcasting - `NaN + anything = NaN`, so missing values must be handled before scaling operations
- Confusing row-wise vs column-wise broadcasting - always double check `.shape` before an operation

### Best Practices
- Always check `.shape` of both arrays before relying on broadcasting, especially with 2D arrays
- Use `np.nanmin()`/`np.nanmax()`/`np.nanmean()` instead of `np.min()`/`np.max()`/`np.mean()` when NaN values are present, to avoid `NaN` propagating into every result
- Handle outliers before normalizing/scaling data, since extreme values distort the whole scaled range


## 4. Vectorization vs Loops

### Introduction
- **Vectorization** means performing an operation on an entire array at once (internally using optimized, compiled C code), instead of writing an explicit Python `for` loop
- Why do we need it?
  - Python loops are slow because each iteration involves Python-level overhead (type checking, function calls, etc.)
  - Vectorized NumPy operations skip this overhead entirely by running the loop inside C

### Real-Life Analogy
- A Python loop is like **stamping 1000 letters by hand**, one at a time, checking and adjusting for each one individually
- A vectorized operation is like using a **printing press** that stamps all 1000 letters simultaneously, using the same fixed setup - drastically faster for large volumes

### Explanation
- NumPy provides **ufuncs** (universal functions) - functions like `np.sqrt()`, `np.exp()`, `+`, `*`, that operate element-wise across entire arrays at C speed
- Whenever you find yourself writing `for item in array: ...` to do simple math, there is almost always a vectorized alternative

### Syntax
```python
# Loop-based (slow)
result = []
for x in arr:
    result.append(x ** 2)

# Vectorized (fast)
result = arr ** 2
```


In [12]:
import time

# Loop-based approach - squaring each account_balance manually
def square_with_loop(arr):
    result = []
    for x in arr:
        result.append(x ** 2)
    return np.array(result)

# Vectorized approach
def square_vectorized(arr):
    return arr ** 2

loop_result = square_with_loop(account_balance)
vectorized_result = square_vectorized(account_balance)

print("Loop result:      ", loop_result)
print("Vectorized result:", vectorized_result)
print("Results match:", np.array_equal(loop_result, vectorized_result))


Loop result:       [1.56250e+02 2.04304e+03 6.22521e+03 2.50000e+01 1.44000e+04 6.25000e+00
 3.60000e+03 0.00000e+00 4.00000e+04 6.40000e+01]
Vectorized result: [1.56250e+02 2.04304e+03 6.22521e+03 2.50000e+01 1.44000e+04 6.25000e+00
 3.60000e+03 0.00000e+00 4.00000e+04 6.40000e+01]
Results match: True


**Line by line explanation:**
- `square_with_loop()` -> manually iterates over each element with a Python `for` loop, appending squared values to a Python list, then converts back to a NumPy array
- `square_vectorized()` -> simply applies `** 2` directly to the whole array - NumPy handles the looping internally in C
- Both give identical results, but their performance differs dramatically at scale (shown next)


In [13]:
# Benchmark on a larger array to clearly see the speed difference
large_array = np.random.uniform(-1000, 1000, 200000)

start = time.time()
loop_result = square_with_loop(large_array)
loop_time = time.time() - start

start = time.time()
vectorized_result = square_vectorized(large_array)
vectorized_time = time.time() - start

print(f"Loop time:       {loop_time:.5f} seconds")
print(f"Vectorized time: {vectorized_time:.5f} seconds")
print(f"Vectorized was approximately {loop_time / vectorized_time:.1f}x faster")


Loop time:       0.04369 seconds
Vectorized time: 0.00000 seconds


ZeroDivisionError: float division by zero

**Line by line explanation:**
- We use 200,000 random values to make the timing difference clearly visible (our 10-row dataset is too small to show a meaningful gap)
- The vectorized version is dramatically faster - often 20-100x or more depending on the machine - because the loop runs inside optimized C code instead of the Python interpreter

### Internal Working
- A Python `for` loop over a NumPy array must, for every single element: fetch the Python object, check its type, perform the operation, wrap the result back into a Python object, and append it - all through the (relatively slow) Python interpreter
- A vectorized ufunc operates directly on the raw contiguous memory block in a single compiled C loop, skipping all that Python-level overhead entirely
- This is the same underlying reason NumPy arrays are faster than Python lists in general (as covered in Notebook 1)

### Time & Space Complexity
- Both loop-based and vectorized squaring are technically O(n) time complexity
- However, vectorized operations have a **much smaller constant factor** (the "hidden multiplier" behind Big-O) because each operation is far cheaper at the C level - this is why real-world timing differences matter even though Big-O looks the same
- Space: both are O(n), assuming output arrays of the same size as input

### Common Mistakes
- Writing manual `for` loops for simple element-wise math when a vectorized alternative exists
- Assuming Big-O complexity alone tells the full performance story - constant factors matter enormously in practice, as shown by our benchmark
- Using Python's built-in `sum()` on a NumPy array instead of `np.sum()` - both work, but `np.sum()` is vectorized and faster

### Best Practices
- Always look for a vectorized NumPy function/operator before writing a manual loop
- Use built-in ufuncs (`np.sqrt`, `np.exp`, `np.log`, arithmetic operators) wherever possible
- Reserve loops for genuinely complex, non-vectorizable logic (e.g., operations depending on previous loop iterations in complex ways)


## 5. NumPy in EDA Workflows

### Introduction
- Before reaching for Pandas, many quick EDA (Exploratory Data Analysis) checks can be done directly with NumPy - aggregations, reshaping, stacking, and missing value handling
- This section ties together everything from Day 3 into one small end-to-end EDA pass on our dataset, using NumPy only

### Explanation - Aggregations
- Common aggregation functions: `np.sum()`, `np.mean()`, `np.median()`, `np.std()`, `np.min()`, `np.max()`
- For arrays with NaN values, use the `nan`-prefixed versions: `np.nansum()`, `np.nanmean()`, `np.nanmedian()`, etc. - these correctly ignore NaN instead of propagating it
- The `axis` parameter controls direction for 2D arrays:
  - `axis=0` -> aggregate **down each column** (across rows)
  - `axis=1` -> aggregate **across each row** (across columns)


In [14]:
# Aggregations - the WRONG way first, to show why nan-versions matter
print("Regular mean of annual_income (with NaN present):", np.mean(annual_income))
print("Correct mean using nanmean:", np.nanmean(annual_income))
print("Correct median using nanmedian:", np.nanmedian(annual_income))

print("\nStandard deviation of account_balance:", np.std(account_balance))
print("Min/Max of tenure_years:", np.min(tenure_years), "/", np.max(tenure_years))


Regular mean of annual_income (with NaN present): nan
Correct mean using nanmean: 168.3125
Correct median using nanmedian: 58.5

Standard deviation of account_balance: 62.23850014259662
Min/Max of tenure_years: 0 / 20


**Line by line explanation:**
- `np.mean(annual_income)` -> returns `nan` because ANY arithmetic involving NaN produces NaN - the whole mean calculation is "contaminated" by just 2 missing values
- `np.nanmean()` / `np.nanmedian()` -> correctly skip NaN values and compute the result from only the valid (non-missing) entries
- This directly demonstrates why we must always check for NaN and use `nan`-prefixed functions during EDA


In [15]:
# axis parameter on our 2D feature_matrix
# columns are: [age, annual_income, account_balance, tenure_years, num_products]

print("feature_matrix shape:", feature_matrix.shape)

# axis=0 -> aggregate DOWN each column (one result per feature)
print("\nMean per feature (axis=0):", np.nanmean(feature_matrix, axis=0))

# axis=1 -> aggregate ACROSS each row (one result per customer)
print("\nSum across features per customer (axis=1):", np.nansum(feature_matrix, axis=1))


feature_matrix shape: (10, 5)

Mean per feature (axis=0): [ 37.8    168.3125  52.71     6.       2.2   ]

Sum across features per customer (axis=1): [  84.5  146.2  134.9   74.5  286.    51.5  180.    99.  1235.    42. ]


**Line by line explanation:**
- `np.nanmean(feature_matrix, axis=0)` -> collapses the **rows** (customers), giving one average per **column** (feature) - 5 results, one per feature
- `np.nansum(feature_matrix, axis=1)` -> collapses the **columns** (features), giving one total per **row** (customer) - 10 results, one per customer
- Mnemonic: `axis=0` moves down the rows (column-wise result), `axis=1` moves across the columns (row-wise result)

> Interview tip: The `axis` parameter confuses almost everyone at first - always double check with a small example like this one if unsure.


### Explanation - Reshaping and Stacking
- `.reshape(new_shape)` -> changes an array's shape without changing its data, as long as the total number of elements matches
- `.flatten()` -> converts any array into a 1D array, always returns a **copy**
- `.ravel()` -> also converts to 1D, but returns a **view** when possible (faster, but be careful of shared memory)
- `np.concatenate()` -> joins arrays along an existing axis
- `np.vstack()` -> stacks arrays vertically (row-wise, adds rows)
- `np.hstack()` -> stacks arrays horizontally (column-wise, adds columns)


In [16]:
# Reshaping example
flat_ages_and_tenure = np.concatenate([age, tenure_years])
print("Concatenated shape:", flat_ages_and_tenure.shape)

reshaped = flat_ages_and_tenure.reshape(2, 10)   # 2 rows, 10 columns
print("\nReshaped into (2, 10):\n", reshaped)

# Flatten back to 1D
flattened_again = reshaped.flatten()
print("\nFlattened back:", flattened_again)


Concatenated shape: (20,)

Reshaped into (2, 10):
 [[25 34 45 29 52 23 41 38 60 31]
 [ 1  3  8  1 15  0  6  4 20  2]]

Flattened back: [25 34 45 29 52 23 41 38 60 31  1  3  8  1 15  0  6  4 20  2]


**Line by line explanation:**
- `np.concatenate([age, tenure_years])` -> joins two 1D arrays of 10 elements each into one array of 20 elements
- `.reshape(2, 10)` -> reorganizes the 20 elements into a `2 x 10` grid - total elements (2*10=20) must match the original count
- `.flatten()` -> collapses back into a single 1D array of 20 elements, as a fresh copy


In [17]:
# Stacking example - combining two new customers into our existing feature_matrix
new_customers = np.array([
    [27, 50.0, 15.0, 2, 1],   # new customer 11
    [48, 80.0, 90.0, 10, 3],  # new customer 12
])

updated_matrix = np.vstack([feature_matrix, new_customers])
print("Original shape:", feature_matrix.shape)
print("Updated shape after vstack:", updated_matrix.shape)


Original shape: (10, 5)
Updated shape after vstack: (12, 5)


**Line by line explanation:**
- `new_customers` -> a small 2D array representing 2 additional customers with the same 5 features
- `np.vstack([feature_matrix, new_customers])` -> stacks them **vertically** (adds rows), so shape goes from `(10, 5)` to `(12, 5)`
- `np.hstack()` would instead be used to add new **columns** (features) to existing rows, requiring the same number of rows to match


### Real-world Example - A small end-to-end NumPy-only EDA pass


In [18]:
print("--- Quick NumPy-only EDA on our FD dataset ---\n")

# 1. Check for missing values
missing_count = np.sum(np.isnan(annual_income))
print(f"1. Missing values in annual_income: {missing_count}")

# 2. Check for outliers using mean vs median comparison
mean_income = np.nanmean(annual_income)
median_income = np.nanmedian(annual_income)
print(f"2. Mean income: {mean_income:.2f} | Median income: {median_income:.2f} -> big gap suggests an outlier")

# 3. Identify the specific outlier customer(s) using boolean masking
outlier_mask = annual_income > (3 * median_income)
print(f"3. Outlier customer IDs: {customer_id[outlier_mask]}")

# 4. Basic groupby-style aggregation using boolean masking (NumPy doesn't have real groupby, Pandas does)
fd_balance_mean = np.nanmean(account_balance[FD == 1])
non_fd_balance_mean = np.nanmean(account_balance[FD == 0])
print(f"4. Avg balance for FD customers: {fd_balance_mean:.2f} | Non-FD: {non_fd_balance_mean:.2f}")

# 5. Normalize account_balance (min-max scaling) using broadcasting
bal_min, bal_max = account_balance.min(), account_balance.max()
normalized_balance = (account_balance - bal_min) / (bal_max - bal_min)
print(f"5. Normalized balances (0-1 scale): {np.round(normalized_balance, 2)}")


--- Quick NumPy-only EDA on our FD dataset ---

1. Missing values in annual_income: 2
2. Mean income: 168.31 | Median income: 58.50 -> big gap suggests an outlier
3. Outlier customer IDs: [9]
4. Avg balance for FD customers: 100.82 | Non-FD: 4.60
5. Normalized balances (0-1 scale): [0.07 0.24 0.4  0.04 0.6  0.   0.31 0.01 1.   0.05]


**Line by line explanation:**
- Step 1 uses `np.isnan()` + `np.sum()` (True counts as 1) to count missing values, exactly as we did with Pandas' `.isnull().sum()`
- Step 2 reuses the mean-vs-median trick from earlier to sense-check for outliers, without needing Pandas' `.describe()`
- Step 3 reuses boolean masking to pinpoint exactly which customer(s) are outliers
- Step 4 shows how **groupby-like behavior** can be approximated in pure NumPy using boolean masks per group (`FD == 1` vs `FD == 0`) - though this becomes much easier and more readable with Pandas `.groupby()` (Day 4), which is precisely why Pandas exists on top of NumPy
- Step 5 reuses broadcasting for normalization, tying together nearly every Day 3 topic in one final pass


## 6. Common Mistakes (Across This Notebook)

- Using `== np.nan` instead of `np.isnan()` to detect missing values
- Forgetting basic slices are views (shared memory) while fancy indexing and boolean masking return copies
- Forgetting to use `nan`-prefixed aggregation functions (`np.nanmean`, `np.nansum`, etc.) when NaN values are present
- Confusing `axis=0` (column-wise result, collapses rows) with `axis=1` (row-wise result, collapses columns)
- Writing manual loops for simple element-wise math instead of using vectorized operations
- Assuming broadcasting works between any two shapes - it only works when dimensions are equal or one of them is 1


## 7. Best Practices (Across This Notebook)

- Always use `np.isnan()` for missing value detection, never `==`
- Use `.copy()` explicitly whenever an independent copy is needed (after slicing especially)
- Default to `nan`-prefixed aggregation functions on any real-world (potentially messy) dataset
- Double-check `.shape` before relying on broadcasting, especially in 2D+ operations
- Always prefer vectorized operations over manual Python loops for performance
- Use NumPy for quick numeric-only EDA checks, but switch to Pandas once mixed-type, labeled, tabular analysis is needed (Day 4)


## 8. Practice Problems (No Solutions)

**Easy**
1. Using slicing, extract the account balances of the first 5 customers
2. Using boolean masking, find all customers who are NOT active (`is_active == 0`)
3. Use `np.nanmean()` to find the average annual_income, correctly ignoring the NaN values

**Medium**
4. Using fancy indexing, extract the ages and cities of customers at positions 2, 5, and 9 (hint: apply the same index list to both arrays)
5. Use broadcasting to convert `account_balance` (assume it's in thousands) into actual currency by multiplying by 1000
6. Use `np.where()` to create an `age_group` array with values `"young"` (age < 30) and `"senior"` (age >= 30)

**Hard**
7. Benchmark a loop-based vs vectorized calculation of `tenure_years ** 2 + num_products` on a large random array of 500,000 elements, and print the speedup factor
8. Using boolean masking and `axis` parameter together, calculate the mean of ALL numeric features (from `feature_matrix`) but only for customers where `FD == 1`
9. Reshape a fresh `np.arange(1, 21)` array into a `(4, 5)` matrix, then use slicing to extract only the middle 2x3 block of that matrix


## 9. Revision Summary

**Indexing & Slicing**
- `arr[i]` single element, `arr[a:b:step]` slice (stop excluded), `matrix[row, col]` for 2D
- Fancy indexing (`arr[[i,j,k]]`) selects specific non-contiguous elements, always returns a copy
- Basic slicing returns a **view** (shares memory); fancy indexing and boolean masking return **copies**

**Boolean Masking & Filtering**
- Conditions on arrays create boolean masks; `array[mask]` filters accordingly
- Combine conditions with `&`, `|`, `~` (not Python's `and`/`or`), each condition in parentheses
- `np.isnan()` (not `==`) correctly detects NaN; `np.where()`/`np.select()` for vectorized conditional logic

**Broadcasting**
- Lets operations happen between different shaped arrays by virtually "stretching" the smaller one
- Compatible if dimensions are equal or one of them is 1, compared right to left
- No extra memory cost - uses stride tricks internally, not actual duplication

**Vectorization vs Loops**
- Vectorized ufuncs run in compiled C code, avoiding slow Python-level loop overhead
- Same Big-O complexity as loops in theory, but vastly better real-world performance due to smaller constant factors

**NumPy in EDA Workflows**
- `nan`-prefixed aggregations (`nanmean`, `nansum`, `nanmedian`) to safely handle missing data
- `axis=0` collapses rows (per-column result); `axis=1` collapses columns (per-row result)
- `.reshape()`, `.flatten()`, `.ravel()` for reshaping; `np.concatenate()`, `np.vstack()`, `np.hstack()` for combining arrays
- Groupby-like behavior is possible with boolean masks in NumPy, but becomes far more natural in Pandas - motivating Day 4

---
*This completes Day 3: NumPy Deep Dive (both Notebook 1: Introduction & Array Creation, and this end-to-end notebook covering Indexing/Slicing, Boolean Masking, Broadcasting, Vectorization, and EDA Workflows).*

*Both Day 3 and Day 4 are now complete. Next: Day 5 - Algorithms for Interviews (Big-O, searching/sorting/hashing, recursion, two-pointer/sliding window, stacks/queues), followed by the full end-of-syllabus 60-question interview Q&A pass per notebook.*
